# spec

> Build a demo spec in Python, check it, and move an old spec to the new schema.

A demo is one JSON object: the spec. It lists blocks, a small state, derived values and data tables.
This module reads the block and fn manifests, gives a builder for each block, and checks a spec with `validate`.

`validate` enforces the think rule: a block that shows a value which a gate's answer depends on must wait for that gate.
So the learner always predicts before the demo shows the result.

In [ ]:
#| default_exp spec

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import json, math, re
from copy import deepcopy
from decimal import Decimal, Context, ROUND_HALF_UP
from pathlib import Path
from fastcore.utils import *
import rupaka

## Manifests

Each block is a folder in `rupaka/blocks/`, with a `block.json` manifest. Each fn has a manifest in `rupaka/fns/`.
A manifest gives the purpose in one line, the props with their types, and the keys. The validator and the agent prompt read the manifests, so they never drift from the code.

In [ ]:
#| export
PKG = Path(rupaka.__file__).parent
SCHEMA = 1
#: props that every block can have
COMMON = ('type', 'after')

@functools.cache
def manifests() -> dict:
    "The block manifests, by name."
    return {p.parent.name: AttrDict(json.loads(p.read_text())) for p in sorted((PKG/'blocks').glob('*/block.json'))}

@functools.cache
def fn_manifests() -> dict:
    "The fn manifests, by name."
    return {p.stem: AttrDict(json.loads(p.read_text())) for p in sorted((PKG/'fns').glob('*.json'))}

Each block folder has its code, its style, its manifest and one example.

In [ ]:
test_eq(set(manifests()), {'axis', 'bars', 'body', 'caption', 'choice', 'force', 'gate', 'grid', 'label3d', 'pixels', 'plot', 'point', 'ranklist', 'readout', 'row',
                          'scene', 'shot', 'slider', 'stack', 'toggle', 'wheel'})
for n in manifests(): assert all((PKG/'blocks'/n/f).exists() for f in (f'{n}.js', f'{n}.css', 'example.json')), n

The fns are `rules`, `rrf`, `softmax`, `lookup`, `expr`, `field`, and four small helpers.

In [ ]:
test_eq(set(fn_manifests()), {'expr', 'field', 'fmt', 'ids', 'lookup', 'pick', 'rrf', 'rules', 'softmax', 'sort'})

## Builders

A builder makes the dict for one block, so a notebook author does not type JSON. A prop with the value `None` is left out.

In [ ]:
#| export
def _b(type, **kw): return {'type': type, **{k: v for k, v in kw.items() if v is not None}}

def from_(path:str) -> dict:
    "Gate options read from a list in the state, for example `from_('join.alts')`."
    return {'from': path}

def is_(v) -> dict:
    "A fixed gate answer."
    return {'is': v}

def demo(id:str, title:str='', ask:str='', state:dict=None, derive=(), blocks=(), tables:dict=None, concept:str=None) -> dict:
    "A whole demo spec."
    d = dict(rupaka=SCHEMA, id=id, concept=concept or id, title=title, ask=ask, state=state or {}, derive=list(derive), blocks=list(blocks))
    if tables: d['tables'] = tables
    return {k: v for k, v in d.items() if v not in ('', None)}

These are the builders for the blocks. Each one takes the props of its manifest. `body` takes `inside` for the `in` prop, because `in` is a Python keyword.

In [ ]:
#| export
def gate(id, prompt, options, answer, why=None, context=None, again=None, hides=None):
    "Predict, then reveal: blocks with `after=id` wait for this gate."
    return _b('gate', id=id, prompt=prompt, options=options, answer=answer, why=why, context=context, again=again, hides=hides)

def readout(text, label=None, **kw): return _b('readout', text=text, label=label, **kw)
def caption(text, label=None, **kw): return _b('caption', text=text, label=label, **kw)
def row(*blocks, **kw): return _b('row', blocks=list(blocks), **kw)
def stack(*blocks, **kw): return _b('stack', blocks=list(blocks), **kw)
def slider(bind, label=None, min=None, max=None, step=None, **kw): return _b('slider', bind=bind, label=label, min=min, max=max, step=step, **kw)
def toggle(bind, label=None, **kw): return _b('toggle', bind=bind, label=label, **kw)
def choice(bind, options, label=None, **kw): return _b('choice', bind=bind, options=list(options), label=label, **kw)
def wheel(bind, items, label=None, **kw): return _b('wheel', bind=bind, items=list(items), label=label, **kw)
def ranklist(bind, label=None, **kw): return _b('ranklist', bind=bind, label=label, **kw)
def bars(bind, label=None, **kw): return _b('bars', bind=bind, label=label, **kw)
def axis(x, *blocks, y=None, **kw): return _b('axis', x=list(x), y=list(y) if y else None, blocks=list(blocks), **kw)
def point(bind=None, **kw): return _b('point', bind=bind, **kw)
def plot(series, **kw): return _b('plot', series=series, **kw)
def grid(bind, label=None, **kw): return _b('grid', bind=bind, label=label, **kw)
def pixels(bind, label=None, **kw): return _b('pixels', bind=bind, label=label, **kw)
def scene(*blocks, bind=None, steps=None, **kw): return _b('scene', bind=bind, steps=list(steps) if steps else None, blocks=list(blocks), **kw)
def body(id, kind, size=None, at=None, inside=None, **kw): return _b('body', id=id, kind=kind, size=size, at=at, **{'in': inside}, **kw)
def force(dir, mag, on=None, at=None, **kw): return _b('force', dir=list(dir), mag=mag, on=on, at=at, **kw)
def label3d(text, on=None, at=None, **kw): return _b('label3d', text=text, on=on, at=at, **kw)
def shot(name=None, bind=None, **kw): return _b('shot', name=name, bind=bind, **kw)

The derive builders name a fn and the key it writes (`out`).

In [ ]:
#| export
def rules(table, ins, out, alts=4): return dict(fn='rules', table=table, **{'in': list(ins)}, out=out, alts=alts)
def rrf(ins, k, out): return dict(fn='rrf', **{'in': list(ins)}, k=k, out=out)
def softmax(src, out, t=None, labels=None, sort=None): return _drop(dict(fn='softmax', **{'in': src}, t=t, labels=labels, sort=sort, out=out))
def lookup(table, key, out, by=None): return _drop(dict(fn='lookup', table=table, key=key, by=by, out=out))
def expr(f, out, over=None): return _drop(dict(fn='expr', expr=f, over=over, out=out))
def field(f, out, n=16, seed=0): return dict(fn='field', expr=f, n=n, seed=seed, out=out)
def fmt(text, out): return dict(fn='fmt', text=text, out=out)
def _drop(d): return {k: v for k, v in d.items() if v is not None}

A small demo: two numbers, a gate on their sum, and a readout that waits for the gate.

In [ ]:
d = demo('sum', 'Sum', state=dict(a=2, b=3), derive=[expr('a + b', 'sum')],
         blocks=[gate('s', 'What is {a} + {b}?', [4, 5, 6], 'sum'), readout('{a} + {b} = {sum}', after='s')])
test_eq(d['blocks'][1], {'type': 'readout', 'text': '{a} + {b} = {sum}', 'after': 's'})
test_eq(d['derive'], [{'fn': 'expr', 'expr': 'a + b', 'out': 'sum'}])

## The expr grammar

`expr` holds a formula in a tiny safe grammar: numbers, names, `+ - * / ^`, parentheses and a few math functions.
The runtime parses it in JavaScript. This is the same parser in Python, so `validate` can check a formula and run it.
There is no `eval` in either one. A name must be a state key, a constant or one of the functions.

In [ ]:
#| export
_TOK = re.compile(r'\s*(?:(\d+\.?\d*(?:[eE][+-]?\d+)?|\.\d+)|([A-Za-z_]\w*)|(\*\*|[-+*/^(),×÷−]))')
_OPS = {'×': '*', '÷': '/', '−': '-', '**': '^'}
def _jsround(x): return math.floor(x + .5)
EXPR_FNS = dict(sin=math.sin, cos=math.cos, tan=math.tan, exp=math.exp, log=math.log, sqrt=math.sqrt, abs=abs,
                min=min, max=max, floor=math.floor, ceil=math.ceil, round=_jsround)
EXPR_CONSTS = dict(pi=math.pi, e=math.e)

def _lex(s):
    out, i = [], 0
    while i < len(s):
        if not s[i:].strip(): break
        m = _TOK.match(s, i)
        if not m: raise ValueError(f'expr: "{s}" has a bad character at {i}')
        i = m.end()
        out.append(('n', float(m[1])) if m[1] else ('id', m[2]) if m[2] else ('op', _OPS.get(m[3], m[3])))
    return out

`_safe` turns a math error into the value JavaScript gives, so both sides agree: `nan` or an infinity.

In [ ]:
#| export
def _safe(f):
    def g(*a):
        try: return f(*a)
        except ZeroDivisionError: return math.copysign(math.inf, a[0]) if a and a[0] else math.nan
        except (ValueError, OverflowError, TypeError): return math.nan
    return g

_BIN = {'+': _safe(lambda a, b: a + b), '-': _safe(lambda a, b: a - b), '*': _safe(lambda a, b: a * b),
        '/': _safe(lambda a, b: a / b), '^': _safe(lambda a, b: math.nan if isinstance(r := a ** b, complex) else r)}

`parse_expr` returns a function of a dict of values, and the set of names that the formula reads.

In [ ]:
#| export
def parse_expr(
    src:str, # the formula
    fns:dict=None # more functions by name, on top of `EXPR_FNS`
):
    "Parse a formula. Return `(f, names)`: `f(env)` gives a number, and `names` are the values it reads."
    ts, p, names, fs = _lex(str(src)), [0], set(), {**EXPR_FNS, **(fns or {})}
    peek = lambda: ts[p[0]] if p[0] < len(ts) else (None, None)
    def eat(v=None):
        t = peek()
        if t[0] is None or (v and t[1] != v): raise ValueError(f'expr: "{src}" needs {v or "more"}')
        p[0] += 1; return t
    def atom():
        k, v = eat()
        if k == 'n': return lambda e: v
        if (k, v) == ('op', '('): a = add(); eat(')'); return a
        if k == 'id':
            if v.startswith('_') or v in ('constructor', 'prototype'): raise ValueError(f'expr: the name "{v}" is not allowed')
            if peek()[1] == '(':
                if v not in fs: raise ValueError(f'expr: no function "{v}"')
                eat('('); args = [add()]
                while peek()[1] == ',': eat(','); args.append(add())
                eat(')'); f = _safe(fs[v])
                return lambda e: f(*[a(e) for a in args])
            if v in EXPR_CONSTS: return lambda e: EXPR_CONSTS[v]
            names.add(v)
            def var(e):
                if v not in e: raise ValueError(f'expr: no value "{v}"')
                return float(e[v])
            return var
        raise ValueError(f'expr: "{src}" has an unexpected "{v}"')
    def pw():
        a = atom()
        if peek()[1] == '^': eat('^'); b = unary(); return lambda e: _BIN['^'](a(e), b(e))
        return a
    def unary():
        if peek()[1] == '-': eat('-'); a = unary(); return lambda e: -a(e)
        if peek()[1] == '+': eat('+'); return unary()
        return pw()
    def binl(nxt, ops):
        def f():
            a = nxt()
            while peek()[0] == 'op' and peek()[1] in ops:
                o, l, r = eat()[1], a, nxt()
                a = (lambda o, l, r: lambda e: _BIN[o](l(e), r(e)))(o, l, r)
            return a
        return f
    mul = binl(unary, '*/'); add = binl(mul, '+-')
    f = add()
    if p[0] < len(ts): raise ValueError(f'expr: "{src}" has extra text at "{ts[p[0]][1]}"')
    return f, names

`2*x^2 - 1` at x = 3 is 17. The power binds before the minus sign, so `-x^2` at x = 3 is −9.

In [ ]:
f, ns = parse_expr('2*x^2 - 1')
test_eq(f(dict(x=3)), 17)
test_eq(ns, {'x'})
test_eq(parse_expr('-x^2')[0](dict(x=3)), -9)
test_eq(parse_expr('max(a, b) + floor(7/2)')[0](dict(a=1, b=4)), 7)

Names that reach into objects, calls of unknown functions, and a second statement are refused.

In [ ]:
for bad in ('constructor', '__proto__', 'alert(1)', 'x;y', 'x.y', '2 +'): test_fail(lambda: parse_expr(bad)[0](dict(x=1, y=2)), contains='expr')

## The fns in Python

Each fn has a Python twin with the same result. `validate` runs the derive list on the opening state with these twins,
so it can check that each gate answer is one of the options. The browser tests compare the two sides.

First, two helpers that copy the runtime: `get` reads a path such as `fused.0.id`, and `show_value` writes a value as the runtime does.

In [ ]:
#| export
_BAD = {'__proto__', 'prototype', 'constructor'}

def get(o, path):
    "Read a path such as `fused.0.id` in dicts and lists; own keys only, as the runtime does."
    for k in str(path).split('.'):
        if o is None or k in _BAD: return None
        if isinstance(o, dict): o = o.get(k)
        elif isinstance(o, (list, tuple)) and re.fullmatch(r'\d+', k): o = o[int(k)] if int(k) < len(o) else None
        else: return None
    return o

def _js_num(x:float) -> str:
    "`Number.prototype.toString()` of `x`: Python's shortest digits, placed by the JavaScript rules."
    if math.isnan(x): return 'NaN'
    if math.isinf(x): return 'Infinity' if x > 0 else '-Infinity'
    if x == 0: return '0'
    _, ds, e = Decimal(repr(abs(x))).normalize().as_tuple()
    ds, n = ''.join(map(str, ds)), len(ds) + e
    k = len(ds)
    s = (ds + '0' * (n - k) if k <= n <= 21 else ds[:n] + '.' + ds[n:] if 0 < n <= 21 else '0.' + '0' * -n + ds if -6 < n <= 0
         else ds[0] + ('.' + ds[1:] if k > 1 else '') + f'e{n - 1:+d}')
    return '-' + s if x < 0 else s

def js_str(v) -> str:
    "`String(v)` in JavaScript, for the values a spec holds."
    if isinstance(v, bool): return 'true' if v else 'false'
    if v is None: return 'null'
    if isinstance(v, (int, float)): return _js_num(float(v))
    if isinstance(v, (list, tuple)): return ','.join('' if x is None else js_str(x) for x in v)
    if isinstance(v, dict): return '[object Object]'
    return str(v)

def show_value(v, d=None) -> str:
    "A value as the runtime shows it: numbers rounded as `toFixed(d)` (default 4) does, with a true minus sign."
    if v is None: return ''
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        x = float(v)
        if math.isfinite(x) and abs(x) < 1e21: x = float(Decimal(x).quantize(Decimal(10) ** -(4 if d is None else d), ROUND_HALF_UP, Context(prec=60)))
        return js_str(x).replace('-', '−', 1)
    if isinstance(v, (list, tuple)): return ', '.join('' if x is None else js_str(x) for x in v)
    return js_str(v)

def fmt_text(s, st) -> str:
    "Fill `{path}` and `{path:2}` from the state, as the runtime does."
    return re.sub(r'\{([\w.]+)(?::(\d))?\}', lambda m: show_value(get(st, m[1]), None if m[2] is None else int(m[2])), str(s or ''))

In [ ]:
st = dict(fused=[dict(id='B', value=0.0328)], k=-60, xs=[1, 2])
test_eq(get(st, 'fused.0.id'), 'B')
test_eq(get(st, 'fused.0.__proto__'), None)
test_eq(fmt_text('{fused.0.id} {fused.0.value:2} {k} {xs}', st), 'B 0.03 −60 1, 2')

`js_str` gives the text of `String(v)` in JavaScript, and `show_value` rounds as `toFixed` does, so a tie such as 1.03125 goes up:

In [ ]:
test_eq([js_str(v) for v in (2, 2.0, 0.1, 1.5, -0.0, 1e21, 1e-7, 0.00001, True, None, 'a', [1, None])],
        ['2', '2', '0.1', '1.5', '0', '1e+21', '1e-7', '0.00001', 'true', 'null', 'a', '1,'])
test_eq([show_value(v) for v in (2.0, -1.5, 1/3, 1.03125, None, True, [1, None])], ['2', '−1.5', '0.3333', '1.0313', '', 'true', '1, '])

Now the fns. `rules` orders its `alts` with the same small hash as the runtime, so both sides give the same options.

In [ ]:
#| export
def _int32(x): x &= 0xffffffff; return x - (1 << 32) if x >= 1 << 31 else x
def _utf16(s): return [c for ch in s for c in ([ord(ch)] if ord(ch) < 0x10000 else list(divmod(ord(ch) - 0x10000, 0x400)))]

def _rules(d, st, spec):
    rows = (spec.get('tables') or {}).get(d['table'])
    if rows is None: raise ValueError(f'rules: no table "{d["table"]}"')
    sub = lambda t: re.sub(r'\{(\w+)\}', lambda m: js_str(st.get(m[1])) if st.get(m[1]) is not None else '', str(t))
    hit = next((r for r in rows if all(st.get(k) in (vs if isinstance(vs, list) else [vs]) for k, vs in (r.get('when') or {}).items())),
               dict(out='?', name='no rule'))
    out = sub(hit['out']); alts = [out]
    for r in rows:
        a = sub(r['out'])
        if a not in alts: alts.append(a)
        if len(alts) >= d.get('alts', 4): break
    seed = 7
    for c in _utf16('|'.join(js_str(st.get(k)) for k in d.get('in', []))): seed = _int32(seed * 31 + c)
    def rank(a):
        h = 5
        for c in _utf16(a): h = _int32(h * 33 + c + seed)
        return h & 1023
    alts.sort(key=lambda a: (rank(a), a))
    return {**hit, 'out': out, 'alts': alts}

`rrf`, `softmax` and `lookup` follow the runtime line by line.

In [ ]:
#| export
def _num(d, st, k, dflt=None):
    v = d.get(k, dflt)
    return float(v if isinstance(v, (int, float)) else st.get(v))

def _rrf(d, st, spec):
    k, acc = _num(d, st, 'k'), {}
    for j, name in enumerate(d['in']):
        for i, id in enumerate(st.get(name) or []):
            acc.setdefault(id, dict(id=id, value=0., parts=[0.] * len(d['in'])))['parts'][j] += 1 / (k + i + 1)
    xs = [{**x, 'value': sum(x['parts'])} for x in acc.values()]
    return sorted(xs, key=lambda x: (-x['value'], x['id']))

def _softmax(d, st, spec):
    xs = get(st, d['in']) or []
    if xs and isinstance(xs[0], list): xs = xs[d.get('row', 0)]
    t = max(1e-6, _num(d, st, 't', 1)); mx = max(xs)
    es = [math.exp((x - mx) / t) for x in xs]; z = sum(es)
    lb = d.get('labels'); lb = lb if isinstance(lb, list) else get(st, lb) if lb else None
    out = [dict(id=js_str(lb[i] if lb else i), value=e / z) for i, e in enumerate(es)]
    return sorted(out, key=lambda x: (-x['value'], x['id'])) if d.get('sort') else out

def _lookup(d, st, spec):
    rows = (spec.get('tables') or {}).get(d['table'])
    if rows is None: raise ValueError(f'lookup: no table "{d["table"]}"')
    v = get(st, d['key'])
    if not d.get('by') and isinstance(v, (int, float)) and not isinstance(v, bool) and float(v).is_integer(): return rows[int(v) % len(rows)]
    return next((r for r in rows if js_str(r.get(d.get('by') or 'id')) == js_str(v)), None)

`expr` gives a number, or a series of points with `over`. `field` gives the rows of an n x n grid, with `noise` from a hash that gives the same bits as the runtime. The small helpers finish the set.

In [ ]:
#| export
def _expr(d, st, spec):
    f = parse_expr(d['expr'])[0]
    if not d.get('over'): return f(st)
    (name, rng), = d['over'].items()
    lo, hi, n = (list(rng) + [101])[:3]
    return [dict(x=(x := lo + (hi - lo) * i / (n - 1)), y=f({**st, name: x})) for i in range(int(n))]

_M32 = 0xFFFFFFFF
def _u32(x): return int(x) & _M32 if math.isfinite(x) else 0
def _mix(h):
    h ^= h >> 16; h = h * 0x85ebca6b & _M32; h ^= h >> 13; h = h * 0xc2b2ae35 & _M32
    return h ^ h >> 16
def _step(h, x): return _mix(((h ^ x) + 0x9e3779b9) & _M32)

def noise(seed:int, i, j) -> float:
    "The `noise(i, j)` of a field with `seed`: mean 0, variance 1, and the same bits as the runtime."
    h = _step(_step(_u32(seed), _u32(i)), _u32(j))
    return (sum(_step(h, k) for k in range(4)) / 4294967296 - 2) * math.sqrt(3)

def _cell_fns(seed=0): return {'noise': lambda i=0, j=0, *_: noise(seed, i, j)}   # a missing argument is 0 and an extra one is ignored, as in the runtime

def _field(d, st, spec):
    n = d.get('n', 16)
    if not (isinstance(n, int) and not isinstance(n, bool) and 1 <= n <= 64): raise ValueError('field: n must be an int from 1 to 64')
    f = parse_expr(d['expr'], _cell_fns(d.get('seed', 0)))[0]
    return [[f({**st, 'i': i, 'j': j}) for j in range(n)] for i in range(n)]

def _sortby(d, st, spec):
    v = lambda x: (x or {}).get(d['by']) if d.get('by') else x
    return sorted(get(st, d['from']) or [], key=v, reverse=bool(d.get('desc')))

FNS = dict(rules=_rules, rrf=_rrf, softmax=_softmax, lookup=_lookup, expr=_expr, field=_field, sort=_sortby,
           ids=lambda d, st, spec: [x.get('id') if isinstance(x, dict) else x for x in get(st, d['from']) or []],
           pick=lambda d, st, spec: [None if x is None else x.get(d['field']) for x in get(st, d['from']) or []],
           fmt=lambda d, st, spec: fmt_text(d['text'], st))

`run` gives the opening state of a demo: the state, then each derive in order. Each gate starts open.

In [ ]:
#| export
def _gates(bs):
    for b in bs or []:
        if b.get('type') == 'gate': yield b
        yield from _gates(b.get('blocks'))

def run(spec, state:dict=None) -> dict:
    "The state of `spec` after the derive list, from its opening state or from `state`."
    st = deepcopy(state if state is not None else spec.get('state') or {})
    st.setdefault('gate', {g['id']: 'open' for g in _gates(spec.get('blocks'))})
    for d in spec.get('derive') or []: st[d['out']] = FNS[d['fn']](d, st, spec)
    return st

The fused scores of the two spike lists at k = 60: B is first, because it is in both lists.

In [ ]:
sp = dict(state=dict(bm25=list('ACEB'), vec=list('DFGB'), k=60), derive=[rrf(['bm25', 'vec'], 'k', 'fused')], blocks=[])
st = run(sp)
test_eq(st['fused'][0]['id'], 'B')
test_close(st['fused'][0]['value'], 1/64 + 1/64)
test_eq(run(sp, {**sp['state'], 'k': 1})['fused'][0]['id'], 'A')

`softmax` at a high temperature makes the shares more equal.

In [ ]:
sp = dict(state=dict(z=[2, 1, 0]), derive=[softmax('z', 'p', t=1), softmax('z', 'q', t=2)], blocks=[])
st = run(sp)
test_close(st['p'][0]['value'], 0.6652, eps=1e-4)
assert st['q'][0]['value'] < st['p'][0]['value']

`field` runs a formula on each cell of an n x n grid. Row `i` and column `j` go from 0 to n − 1. This formula fades a ramp into `noise` as `t` goes from 0 to 1000:

In [ ]:
DIFF = 'sqrt(1 - t/1000) * (i + j)/30 + sqrt(t/1000) * noise(i, j)'
def fld(f, t=500, seed=7, n=16): return run(dict(state=dict(t=t), derive=[field(f, 'img', n=n, seed=seed)]))['img']
img = fld(DIFF)
test_eq((len(img), len(img[0])), (16, 16))
[round(v, 3) for v in img[0][:4]]

The same seed gives the same field, and another seed gives another field. At t = 0 there is no noise, so each cell is (i + j) / 30.

In [ ]:
test_eq(fld(DIFF), img)
assert fld(DIFF, seed=8) != img
test_eq(fld(DIFF, t=0)[3][4], 7/30)
fld(DIFF, t=0)[3][:4]

## validate

`validate(spec)` returns a list of errors. Each error has a path and one sentence, for example `blocks[3].bind: "fused" is not in state or derive`.
An empty list means the spec is good. It checks, in order:

1. The shape: the schema number, the top-level keys, and a list of blocks.
2. Each derive: the fn exists, its props have the right types, and it reads only keys that come before it.
3. Each block: the type exists, its props have the right types, the required props are there, and each `bind` names a key.
   Each `after` and each `lock` names a gate, and no gate waits for itself. A block above its gate can wait for it, for example a mark on a figure.
4. Each gate: its answer resolves on the opening state and is one of its options. A gate whose answer a live control can change names its inputs with `context`.
5. The think rule (below).
6. Text: each sentence has at most 20 words.

First, the prop types. A scene adds four: `vec3` ([x, y, z]), `vec3key` ([x, y, z], where an item can be a state key), `nums` (one to three sizes) and `texts` (lines of text). `_reads` gives the keys that a value of a type reads, so the think rule can follow them.

In [ ]:
#| export
_TPL = re.compile(r'\{([\w.]+)(?::\d)?\}')
_SPECIAL = {'gate', 'log', '_'}
def _root(path): return str(path).split('.')[0]

def _type_err(t, v, tables):
    "A sentence when `v` is not of prop type `t`, else None."
    num = lambda x: isinstance(x, (int, float)) and not isinstance(x, bool)
    ok = {'str': lambda: isinstance(v, str), 'text': lambda: isinstance(v, str), 'num': lambda: num(v),
          'int': lambda: isinstance(v, int) and not isinstance(v, bool), 'bool': lambda: isinstance(v, bool),
          'list': lambda: isinstance(v, list), 'dict': lambda: isinstance(v, dict), 'any': lambda: True,
          'key': lambda: isinstance(v, str), 'path': lambda: isinstance(v, str), 'expr': lambda: isinstance(v, str),
          'cellexpr': lambda: isinstance(v, str),
          'vec3': lambda: isinstance(v, list) and len(v) == 3 and all(map(num, v)),
          'vec3key': lambda: isinstance(v, list) and len(v) == 3 and all(num(x) or isinstance(x, str) for x in v),
          'nums': lambda: isinstance(v, list) and 1 <= len(v) <= 3 and all(num(x) and x > 0 for x in v),
          'texts': lambda: isinstance(v, list) and all(isinstance(x, str) for x in v),
          'keys': lambda: isinstance(v, list) and all(isinstance(x, str) for x in v),
          'numkey': lambda: num(v) or isinstance(v, str),
          'options': lambda: (isinstance(v, list) and len(v) >= 2) or (isinstance(v, dict) and set(v) == {'from'}),
          'answer': lambda: isinstance(v, str) or (isinstance(v, dict) and set(v) == {'is'}),
          'range': lambda: isinstance(v, list) and len(v) == 2 and all(map(num, v)) and v[0] < v[1],
          'blocks': lambda: isinstance(v, list) and all(isinstance(x, dict) for x in v),
          'table': lambda: isinstance(v, str) and v in tables,
          'over': lambda: isinstance(v, dict) and len(v) == 1 and isinstance(r := first(v.values()), list) and 2 <= len(r) <= 3 and all(map(num, r))}
    if t not in ok: return f'the manifest names an unknown type "{t}"'
    if ok[t](): return None
    return {'table': f'"{v}" is not a table in tables', 'options': 'must be a list of two or more options, or {"from": path}',
            'answer': 'must be a state path, or {"is": value}', 'range': 'must be [min, max] with min < max',
            'over': 'must be {"x": [min, max, n]}', 'vec3': 'must be [x, y, z]: three numbers',
            'vec3key': 'must be [x, y, z]: three numbers or state keys', 'nums': 'must be a list of one to three numbers above 0',
            'texts': 'must be a list of lines of text'}.get(t, f'must be of type {t}')

def _reads(t, v) -> set:
    "The root keys that a prop value of type `t` reads."
    if t in ('key', 'path'): return {_root(v)}
    if t == 'keys': return {_root(x) for x in v}
    if t == 'numkey': return {v} if isinstance(v, str) else set()
    if t == 'text': return {_root(m) for m in _TPL.findall(v)}
    if t == 'texts': return {_root(m) for x in v for m in _TPL.findall(x)}
    if t == 'vec3key': return {_root(x) for x in v if isinstance(x, str)}
    if t == 'options': return {_root(v['from'])} if isinstance(v, dict) else set()
    if t == 'answer': return {_root(v)} if isinstance(v, str) else set()
    if t == 'expr':
        try: return parse_expr(v)[1]
        except ValueError: return set()
    if t == 'cellexpr':
        try: return parse_expr(v, _cell_fns())[1] - {'i', 'j'}
        except ValueError: return set()
    return set()

`_props` checks the props of one block or one derive entry against its manifest. A prop with `min` or `max` in its manifest must be in that range. A prop with `of` must be one of the values in that list. It returns the keys that the entry reads.

In [ ]:
#| export
def _props(m, x, path, known, tables, err, skip=()):
    props, reads = m.props, set()
    for k, p in props.items():
        if p.get('req') and k not in x: err(f'{path}', f'needs "{k}" ({p.get("doc") or p["type"]})')
    for k, v in x.items():
        if k in skip: continue
        if k not in props: err(f'{path}.{k}', f'is not a prop of {m.name}; use one of: {", ".join(props)}'); continue
        t = props[k]['type']
        if (e := _type_err(t, v, tables)): err(f'{path}.{k}', e); continue
        if 'of' in props[k] and v not in props[k]['of']: err(f'{path}.{k}', f'"{v}" is not one of: {", ".join(props[k]["of"])}'); continue
        lo, hi = props[k].get('min', -math.inf), props[k].get('max', math.inf)
        if ('min' in props[k] or 'max' in props[k]) and not lo <= v <= hi: err(f'{path}.{k}', f'must be from {lo} to {hi}'); continue
        rs = _reads(t, v)
        for r in sorted(rs - known - _SPECIAL - (set(x['over']) if t == 'expr' and x.get('over') else set())):
            err(f'{path}.{k}', f'"{r}" is not in state or derive')
        if t == 'expr' and x.get('over'): rs -= set(x['over'])
        if t in ('expr', 'cellexpr'):
            try: parse_expr(v, _cell_fns() if t == 'cellexpr' else None)
            except ValueError as e: err(f'{path}.{k}', str(e))
        reads |= rs
    for grp in m.get('one_of', []):
        if sum(g in x for g in grp) != 1: err(path, f'needs exactly one of {", ".join(grp)}')
    return reads

`_walk` visits every block with its path and its parent, in reading order.

In [ ]:
#| export
def _walk(bs, path='blocks', parent=None):
    for i, b in enumerate(bs or []):
        p = f'{path}[{i}]'
        yield p, b, parent
        if isinstance(b, dict) and isinstance(b.get('blocks'), list): yield from _walk(b['blocks'], f'{p}.blocks', b)

### The think rule

The answer of a gate has a root key: for `fused.0.id`, it is `fused`. A block must wait for the gate when it reads a key that gives the answer away:

- the root key itself;
- each derived key that the root is computed from (raw inputs are the question, so they stay visible);
- each derived key that is computed from any of these.

A fixed answer (`{"is": ...}`) has no root key, so the gate names the keys that give it away in `hides`.
A block waits for the gate when it, or a block that holds it, has `after` set to that gate, or to a gate that itself waits for it.

In [ ]:
#| export
def _taint(g, deps, state_keys):
    "The keys that give away the answer of gate `g`."
    a = g.get('answer')
    roots = {_root(a)} if isinstance(a, str) else set(g.get('hides') or [])
    out, todo = set(roots), list(roots)
    while todo:
        k = todo.pop()
        for r in deps.get(k, ()):
            if r not in state_keys and r not in out: out.add(r); todo.append(r)
    grew = True
    while grew:
        grew = False
        for k, rs in deps.items():
            if k not in out and rs & out: out.add(k); grew = True
    return out

def _ancestors(k, deps):
    "All keys that `k` is computed from, raw inputs too."
    out, todo = set(), [k]
    while todo:
        for r in deps.get(todo.pop(), ()):
            if r not in out: out.add(r); todo.append(r)
    return out

### Text

ASD-STE100 asks for sentences of at most 20 words. This cheap check counts the words in each sentence of each text prop.

In [ ]:
#| export
MAX_WORDS = 20
def long_sentences(s:str) -> list:
    "The sentences of `s` that have more than `MAX_WORDS` words."
    return [x for x in re.split(r'(?<=[.?!])\s+', str(s)) if len(re.findall(r"[\w’'{}.:-]+", x)) > MAX_WORDS]

In [ ]:
test_eq(long_sentences('Short one. ' + 'word ' * 21 + 'end.'), ['word ' * 21 + 'end.'])

### validate

Now `validate` puts the checks together.

In [ ]:
#| export
TOP = ('rupaka', 'id', 'concept', 'title', 'ask', 'state', 'derive', 'blocks', 'tables')
_TEXT_TYPES = ('str', 'text')

def validate(spec) -> L:
    "The errors of `spec`, each as `path: sentence`. An empty list means the spec is good."
    errs = L()
    err = lambda p, m: errs.append(f'{p}: {m}')
    if not isinstance(spec, dict): return L(['spec: must be a JSON object'])
    if spec.get('rupaka', SCHEMA) not in (SCHEMA, SCHEMA - 1): err('rupaka', f'schema {spec.get("rupaka")} is not supported; use {SCHEMA}')
    for k in spec:
        if k not in TOP: err(k, f'is not a top-level key; use one of: {", ".join(TOP)}')
    if not isinstance(spec.get('id'), str) or not spec.get('id'): err('id', 'needs a short id string')
    st, tables = spec.get('state') or {}, spec.get('tables') or {}
    if not isinstance(st, dict): err('state', 'must be an object'); st = {}
    if not isinstance(tables, dict): err('tables', 'must be an object'); tables = {}
    if not isinstance(spec.get('blocks'), list) or not spec['blocks']: err('blocks', 'needs a list of blocks'); return errs
    for k in ('ask', 'title'):
        for s in long_sentences(spec.get(k, '')): err(k, f'this sentence has more than {MAX_WORDS} words: "{s}"')
        if _TPL.search(str(spec.get(k, ''))): err(k, 'is plain text, so {path} shows as it is; write the value, or put it in a readout')
    known, deps, fms = set(st), {}, fn_manifests()
    for i, d in enumerate(spec.get('derive') or []):
        p = f'derive[{i}]'
        if not isinstance(d, dict) or d.get('fn') not in fms: err(f'{p}.fn', f'"{d.get("fn") if isinstance(d, dict) else d}" is not a fn; use one of: {", ".join(fms)}'); continue
        if not isinstance(d.get('out'), str) or not d['out']: err(f'{p}.out', 'needs the key to write'); continue
        if d['out'] in st or d['out'] in _SPECIAL: err(f'{p}.out', f'"{d["out"]}" is a state key or a reserved name; use a new key')
        deps[d['out']] = _props(fms[d['fn']], d, p, known, tables, err, skip=('fn', 'out'))
        known.add(d['out'])
    if errs: return errs
    try: st0 = run(spec)
    except Exception as e: err('derive', f'the opening state does not compute: {e}'); return errs
    return errs + _check_blocks(spec, st, st0, known, deps, tables)

`_check_blocks` does the block checks, the gate checks and the think rule. It gives each scene to `_check_scene` (below).

In [ ]:
#| export
def _check_blocks(spec, st, st0, known, deps, tables) -> L:
    errs, ms = L(), manifests()
    err = lambda p, m: errs.append(f'{p}: {m}')
    gates, waits, controls = {}, {}, []      # gate id -> (path, block); gate id -> gates it waits for; (path, block, keys it writes)
    reads = {}
    for p, b, parent in _walk(spec['blocks']):
        if not isinstance(b, dict) or b.get('type') not in ms:
            err(f'{p}.type', f'"{b.get("type") if isinstance(b, dict) else b}" is not a block; use one of: {", ".join(ms)}'); continue
        m = ms[b['type']]
        if m.get('parent') and (parent or {}).get('type') != m.parent: err(p, f'a {b["type"]} must be inside a {m.parent}')
        only = m.props.get('blocks', {}).get('only')
        if only: [err(f'{p}.blocks[{j}]', f'a {m.name} holds only {", ".join(only)}') for j, c in enumerate(b.get('blocks') or []) if c.get('type') not in only]
        skip = ('type', 'after', 'blocks') + (('lock',) if m.get('lockable') else ())
        reads[p] = _props(m, b, p, known, tables, err, skip=skip)
        if 'lock' in b and not m.get('lockable'): err(f'{p}.lock', f'a {m.name} has no controls to lock')
        if b['type'] == 'scene': _check_scene(p, b, st, st0, err)
        for k, t in ((k, m.props[k]['type']) for k in b if k in m.props):
            for x in (b[k] if t == 'texts' else [b[k]] if t in _TEXT_TYPES else []):
                for s in long_sentences(x): err(f'{p}.{k}', f'this sentence has more than {MAX_WORDS} words: "{s}"')
            if t == 'str' and _TPL.search(str(b[k])): err(f'{p}.{k}', 'is plain text, so {path} shows as it is; use a text prop such as a caption')
        if b['type'] == 'gate':
            if b['id'] in gates: err(f'{p}.id', f'the gate id "{b["id"]}" is used twice')
            gates[b['id']] = (p, b)
        if m.get('lockable'): controls.append((p, b, {b[k] for k in ('bind', 'angle', 'x', 'y') if isinstance(b.get(k), str)} & set(st)))
    allp = {p: b for p, b, _ in _walk(spec['blocks']) if isinstance(b, dict)}
    for p, b in allp.items():
        for k in ('after', 'lock'):
            if k in b and b[k] not in gates: err(f'{p}.{k}', f'"{b[k]}" is not a gate')
    if not gates: err('blocks', 'needs at least one gate: the learner predicts first')
    for gid, (p, g) in gates.items(): waits[gid] = _waits(p, allp)
    for gid, (p, g) in gates.items():
        if gid in _all_waits(p, allp, waits): err(f'{p}.after', f'the gate "{gid}" waits for itself')
    for gid, (p, g) in gates.items(): errs += _check_gate(gid, p, g, st, st0, deps, allp, waits, controls, reads)
    return errs

def _waits(p, allp):
    "The gates that block `p` waits for: its own `after` and the `after` of each block that holds it."
    return {b['after'] for q, b in allp.items() if (p == q or p.startswith(q + '.')) and b.get('after')}

`_check_gate` checks one gate: its answer, its options, its context, and the think rule for its answer.

In [ ]:
#| export
def _all_waits(p, allp, waits):
    "Every gate that block `p` waits for, also through the gates that it waits for."
    out, todo = set(), list(_waits(p, allp))
    while todo:
        g = todo.pop()
        if g not in out: out.add(g); todo += list(waits.get(g, ()))
    return out

def _check_gate(gid, p, g, st, st0, deps, allp, waits, controls, reads) -> L:
    errs = L()
    err = lambda q, m: errs.append(f'{q}: {m}')
    a = g.get('answer')
    ans = a['is'] if isinstance(a, dict) else get(st0, a)
    opts = g['options'] if isinstance(g.get('options'), list) else get(st0, g['options']['from'])
    if ans is None: err(f'{p}.answer', f'"{a}" gives no value on the opening state')
    elif not isinstance(opts, list): err(f'{p}.options', f'"{g["options"]["from"]}" is not a list on the opening state')
    elif js_str(ans) not in [js_str(o) for o in opts]: err(f'{p}.answer', f'the answer "{js_str(ans)}" is not one of the options {[js_str(o) for o in opts]}')
    taint = _taint(g, deps, set(st))
    if (reads.get(p, set()) - _reads('options', g['options']) - _reads('answer', a) - _reads('text', g.get('context', '')) - set(g.get('hides') or [])) & taint:
        err(f'{p}.prompt', 'the prompt shows a value that gives the answer away')
    for q, b in allp.items():
        if q == p: continue
        if (rs := reads.get(q, set()) & taint) and gid not in _all_waits(q, allp, waits):
            k = sorted(rs)[0]
            err(q, f'this {b["type"]} shows "{k}", which gives away the answer of gate "{gid}"; add "after": "{gid}"')
    if isinstance(a, str) and not g.get('context'):
        need = _ancestors(_root(a), deps) | {_root(a)}
        for q, b, ks in controls:
            if ks & need: err(f'{p}.context', f'the {b["type"]} at {q} changes the answer, so name the inputs, for example "For {{{sorted(ks & need)[0]}}}:"'); break
    return errs

### Scenes

A `scene` holds `body`, `force`, `label3d` and `shot` blocks. The manifests check each prop alone. `_check_scene` checks how the parts fit together:

- Each body id is used one time. A body that is `in` another body names a body above it, so a body cannot hold itself.
- A `size` has the right count of numbers for its `kind`.
- A force or a label that is `on` a body names a body of the same scene.
- A shot name is `wide`, `side`, `front`, `top` or `close:<body id>`. A shot that binds a key names a state key with a good shot name, so a host can set it.
- A part with a `step` is in a scene with `bind`, and the step is in the range of the `steps` lines.
- The step key is a state key with an int from 1. There are at most 8 steps, 40 bodies, 12 forces and 12 labels, and a chip has at most 60 characters.

In [ ]:
#| export
SHOTS = ('wide', 'side', 'front', 'top')
_SIZES = dict(box=(3,), cylinder=(2, 3), cone=(2,), sphere=(1,), capsule=(2,), torus=(2,), group=(0,))
_MOST = dict(body=40, force=12, label3d=12)

def shot_error(v, ids) -> str|None:
    "Why `v` is not a shot name for a scene with the body `ids`, or None."
    if isinstance(v, str) and (v in SHOTS or (v.startswith('close:') and v[6:] in ids)): return None
    if isinstance(v, str) and v.startswith('close:'): return f'"{v[6:]}" is not a body in this scene, so "{v}" is not a shot'
    return f'"{v}" is not a shot; use wide, side, front, top or close:<body id>'

def _check_scene(p, b, st, st0, err):
    kids = [(f'{p}.blocks[{i}]', k) for i, k in enumerate(b.get('blocks') or []) if isinstance(k, dict)]
    ids = []
    for q, k in kids:
        if k.get('type') != 'body': continue
        if k.get('id') in ids: err(f'{q}.id', f'the body id "{k["id"]}" is used twice')
        if 'in' in k and k['in'] not in ids: err(f'{q}.in', f'"{k["in"]}" is not a body above this one in the scene')
        want = _SIZES.get(k.get('kind'))
        if want and isinstance(k.get('size'), list) and len(k['size']) not in want:
            err(f'{q}.size', f'a {k["kind"]} has no size' if want == (0,) else f'a {k["kind"]} needs {" or ".join(map(str, want))} numbers')
        ids.append(k.get('id'))
    for t, most in _MOST.items():
        if sum(k.get('type') == t for _, k in kids) > most: err(p, f'a scene has at most {most} {t} blocks')
    steps = b.get('steps') if isinstance(b.get('steps'), list) else None
    if steps is not None:
        if len(steps) > 8: err(f'{p}.steps', 'a scene has at most 8 steps')
        for j, x in enumerate(steps):
            if isinstance(x, str) and len(x) > 200: err(f'{p}.steps[{j}]', 'a step line has at most 200 characters')
    if 'bind' in b:
        v = get(st0, b['bind'])
        if b['bind'] not in st: err(f'{p}.bind', f'"{b["bind"]}" must be a state key, so that the learner and the host can set the step')
        elif not (isinstance(v, int) and not isinstance(v, bool) and v >= 1): err(f'{p}.bind', f'the step "{b["bind"]}" must start as an int from 1')
        elif steps and v > len(steps): err(f'{p}.bind', f'the step starts at {v}, but there are {len(steps)} steps')
    for q, k in kids:
        if 'on' in k and k['on'] not in ids:
            err(f'{q}.on', f'"{k["on"]}" is not a body in this scene' + (f'; use one of: {", ".join(map(str, ids))}' if ids else ''))
        if 'step' in k:
            if 'bind' not in b: err(f'{q}.step', 'a step needs a scene with bind')
            elif steps and isinstance(k['step'], int) and k['step'] > len(steps): err(f'{q}.step', f'the scene has {len(steps)} steps')
        for t in ('text', 'label'):
            if k.get('type') in ('label3d', 'force') and isinstance(k.get(t), str) and len(k[t]) > 60: err(f'{q}.{t}', 'a chip has at most 60 characters')
        if k.get('type') != 'shot': continue
        if 'name' in k and (e := shot_error(k['name'], ids)): err(f'{q}.name', e)
        if 'bind' in k:
            if k['bind'] not in st: err(f'{q}.bind', f'"{k["bind"]}" must be a state key, so that a host can set the shot')
            elif (e := shot_error(get(st0, k['bind']), ids)): err(f'{q}.bind', f'the opening value: {e}')

`shots` gives the shot names that the scenes of a spec know. A host checks the `cam` operation of a board with it.

In [ ]:
#| export
def shots(spec) -> list:
    "The shot names that the scenes of `spec` know, for the board's `cam` operation."
    ids = [b['id'] for _, b, parent in _walk(spec.get('blocks')) if b.get('type') == 'body' and (parent or {}).get('type') == 'scene']
    return [*SHOTS, *(f'close:{i}' for i in ids)] if any(b.get('type') == 'scene' for _, b, _ in _walk(spec.get('blocks'))) else []

### Tests for validate

A good demo has no errors.

In [ ]:
good = demo('sum', 'Sum', state=dict(a=2, b=3), derive=[expr('a + b', 'sum')],
            blocks=[gate('s', 'What is {a} + {b}?', [4, 5, 6], 'sum'), readout('{a} + {b} = {sum}', after='s')])
test_eq(validate(good), [])

A readout of the sum before the gate fails the think rule, with the path of the readout.

In [ ]:
bad = deepcopy(good); del bad['blocks'][1]['after']
test_eq(validate(bad), ['blocks[1]: this readout shows "sum", which gives away the answer of gate "s"; add "after": "s"'])

An unknown block, an unknown fn, an unknown prop and a missing prop each fail with a path.

In [ ]:
def errs_of(**kw): return validate({**deepcopy(good), **kw})
test_eq(errs_of(blocks=[*good['blocks'], dict(type='dial')])[0][:23], 'blocks[2].type: "dial" ')
assert errs_of(derive=[dict(fn='sum', out='sum')])[0].startswith('derive[0].fn: "sum" is not a fn')
assert errs_of(blocks=[good['blocks'][0], {**good['blocks'][1], 'colour': 'red'}])[0].startswith('blocks[1].colour: is not a prop of readout')
assert 'blocks[1]: needs "text"' in validate({**good, 'blocks': [good['blocks'][0], dict(type='readout', after='s')]})[0]

A bind to a key that is not in the state fails. So does an `after` that names no gate.

In [ ]:
assert 'blocks[2].bind: "k" is not in state or derive' in errs_of(blocks=[*good['blocks'], slider('k')])
test_eq(errs_of(blocks=[readout('{sum}', after='t'), good['blocks'][0]]), ['blocks[0].after: "t" is not a gate', 'blocks[0]: this readout shows "sum", which gives away the answer of gate "s"; add "after": "s"'])

A `field` needs an `n` from 1 to 64 and an int seed. A bad formula fails with its `expr:` error. The names `i`, `j` and `noise` need no state key.

In [ ]:
fd = demo('fd', state=dict(t=500), derive=[field(DIFF, 'img', n=16, seed=7)],
          blocks=[gate('g', 'Which corner is largest?', ['top left', 'bottom right'], is_('bottom right'), hides=['img']), pixels('img', 'Grid at t = {t}', after='g')])
def field_errs(**kw): return validate({**fd, 'derive': [{**fd['derive'][0], **kw}]})
test_eq(validate(fd), [])
test_eq([field_errs(n=v) for v in (0, 65, '16')], [['derive[0].n: must be from 1 to 64'], ['derive[0].n: must be from 1 to 64'], ['derive[0].n: must be of type int']])
test_eq(field_errs(seed='7'), ['derive[0].seed: must be of type int'])
test_eq(field_errs(expr='i +* j'), ['derive[0].expr: expr: "i +* j" has an unexpected "*"'])
field_errs(expr='i +* j')

A gate answer must be one of its options. A demo needs a gate.

In [ ]:
test_eq(errs_of(blocks=[gate('s', 'Sum?', [1, 2], 'sum')]), ['blocks[0].answer: the answer "5" is not one of the options [\'1\', \'2\']'])
test_eq(errs_of(blocks=[readout('{a}')]), ['blocks: needs at least one gate: the learner predicts first'])

A live control that changes the answer needs a `context` on the gate, so the verdict names its inputs.

In [ ]:
live = errs_of(blocks=[slider('a', lock='s'), *good['blocks']])
test_eq(live, ['blocks[1].context: the slider at blocks[0] changes the answer, so name the inputs, for example "For {a}:"'])
ok = deepcopy(good); ok['blocks'] = [slider('a', lock='s'), {**good['blocks'][0], 'context': 'For {a} + {b}:'}, good['blocks'][1]]
test_eq(validate(ok), [])

The think rule follows derived keys both ways. Here `twice` is computed from the answer `sum`, so a readout of `twice` must wait too.

In [ ]:
t = deepcopy(good); t['derive'].append(expr('2 * sum', 'twice')); t['blocks'].append(readout('{twice}'))
test_eq(validate(t), ['blocks[2]: this readout shows "twice", which gives away the answer of gate "s"; add "after": "s"'])

A block inside a `stack` that waits for the gate passes, because the stack waits.

In [ ]:
t['blocks'][2] = stack(readout('{twice}'), after='s')
test_eq(validate(t), [])

A fixed answer names the keys that give it away in `hides`.

In [ ]:
h = demo('h', state=dict(m=1), derive=[expr('m * x', 'line', over=dict(x=[-1, 1, 3]))],
         blocks=[gate('g', 'Steeper or flatter?', ['steeper', 'flatter'], is_('steeper'), hides=['line']),
                 axis([-1, 1], plot('line'), y=[-3, 3])])
test_eq(validate(h), ['blocks[1].blocks[0]: this plot shows "line", which gives away the answer of gate "g"; add "after": "g"'])

A plain-text prop, such as `ask` or a label, cannot hold `{path}`: the runtime would show the braces.

In [ ]:
assert errs_of(ask='The list has {a} items.')[0].startswith('ask: is plain text')
assert errs_of(blocks=[*good['blocks'], readout('{a}', label='a is {a}')])[0].startswith('blocks[2].label: is plain text')

Each sentence has at most 20 words.

In [ ]:
assert 'more than 20 words' in errs_of(ask=' '.join(['word'] * 25) + '.')[0]

### Tests for scenes

A small scene: a box and a cone, a force on the box, a label, and a shot that a host can set. The force shows the answer, so it waits for the gate.

In [ ]:
sc = demo('lift', 'Lift', state=dict(step=1, cam='wide', w=4), derive=[expr('w * 2', 'up')],
          blocks=[gate('g', 'What is twice {w}?', [6, 8], 'up'),
                  scene(body('base', 'box', [2, 1, 2]), body('top', 'cone', [0.6, 1], [0, 1, 0], inside='base'),
                        force([0, 1, 0], 'up', on='base', at=[0, 1.5, 0], label='lift, {up}', after='g'),
                        label3d('the base', on='base'), shot(bind='cam'), shot('close:top', step=2),
                        bind='step', steps=['A box with a cone.', 'Look at the cone.'])])
test_eq(validate(sc), [])
test_eq(shots(sc), ['wide', 'side', 'front', 'top', 'close:base', 'close:top'])

A helper puts one bad part in the scene, and gives the errors.

In [ ]:
def scene_errs(i, **kw):
    s = deepcopy(sc); part = s['blocks'][1]['blocks'][i]
    for k, v in kw.items():
        if v is None: part.pop(k, None)
        else: part[k] = v
    return list(validate(s))

An unknown body kind, a force on a missing body, and a bad shot name are each refused with a clear sentence.

In [ ]:
test_eq(scene_errs(0, kind='pyramid'), ['blocks[1].blocks[0].kind: "pyramid" is not one of: box, cylinder, cone, sphere, capsule, torus, group'])
test_eq(scene_errs(2, on='wing'), ['blocks[1].blocks[2].on: "wing" is not a body in this scene; use one of: base, top'])
test_eq(scene_errs(5, name='close:wing'), ['blocks[1].blocks[5].name: "wing" is not a body in this scene, so "close:wing" is not a shot'])
test_eq(scene_errs(5, name='zoom'), ['blocks[1].blocks[5].name: "zoom" is not a shot; use wide, side, front, top or close:<body id>'])

A bound shot must start with a good name. A body cannot be in a body below it, and a size needs the right count of numbers.

In [ ]:
bad = deepcopy(sc); bad['state']['cam'] = 'far'
test_eq(validate(bad), ['blocks[1].blocks[4].bind: the opening value: "far" is not a shot; use wide, side, front, top or close:<body id>'])
test_eq(scene_errs(0, **{'in': 'top'}), ['blocks[1].blocks[0].in: "top" is not a body above this one in the scene'])
test_eq(scene_errs(1, size=[1, 2, 3]), ['blocks[1].blocks[1].size: a cone needs 2 numbers'])
assert 'must be [x, y, z]' in scene_errs(1, at=[0, 1])[0]

A part goes only in a scene. A part with a step needs the step key, and the step must be in the range of the lines.

In [ ]:
test_eq(errs_of(blocks=[*good['blocks'], body('b', 'box')]), ['blocks[2]: a body must be inside a scene'])
test_eq(scene_errs(5, step=3), ['blocks[1].blocks[5].step: the scene has 2 steps'])
nb = deepcopy(sc); del nb['blocks'][1]['bind']
assert 'blocks[1].blocks[5].step: a step needs a scene with bind' in validate(nb)

The think rule holds in a scene too. The force shows `up`, the answer of the gate, so it must wait for the gate.

In [ ]:
test_eq(scene_errs(2, after=None), ['blocks[1].blocks[2]: this force shows "up", which gives away the answer of gate "g"; add "after": "g"'])

## upgrade

`upgrade(spec)` moves a spec from schema N−1 to N. Schema 1 is the first, so today the step is the identity. The frame and the test are in place for schema 2.

In [ ]:
#| export
UPGRADES = {0: lambda s: s}   #: schema n -> a function that moves a spec from n to n + 1

def upgrade(spec:dict) -> dict:
    "A copy of `spec` at the current schema."
    s = deepcopy(spec)
    v = s.get('rupaka', SCHEMA)
    if v > SCHEMA: raise ValueError(f'schema {v} is newer than this rupaka ({SCHEMA}); update rupaka')
    while v < SCHEMA: s = UPGRADES[v](s); v += 1
    s['rupaka'] = SCHEMA
    return s

In [ ]:
test_eq(upgrade({**good, 'rupaka': 0}), good)
test_fail(lambda: upgrade({**good, 'rupaka': 9}), contains='newer')

## The block examples

Each block's `example.json` must pass `validate`. The browser tests draw each one.

In [ ]:
#| export
def examples() -> dict:
    "The example spec of each block, by block name."
    return {n: json.loads((PKG/'blocks'/n/'example.json').read_text()) for n in manifests()}

In [ ]:
for n, ex in examples().items(): test_eq((n, validate(ex)), (n, []))

## The gallery

Hand-made demos live in `rupaka/demos/`. The agent shows two of them to the model as examples, and abhyasa can import them.

In [ ]:
#| export
def gallery() -> dict:
    "The gallery demos, by id."
    return {(s := json.loads(p.read_text()))['id']: s for p in sorted((PKG/'demos').glob('*.json'))}

Each gallery demo passes `validate`.

In [ ]:
test_eq({k: list(validate(s)) for k, s in gallery().items()}, {k: [] for k in gallery()})
assert {'vowel-sandhi', 'reciprocal-rank-fusion', 'softmax-temperature', 'nakshatra-pada', 'parabola-vertex'} <= set(gallery())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()